# 🚀 PaliGemma Math Recognition Training (A100 Optimized)

**High-performance training pipeline optimized for A100 GPUs**

**Key Optimizations:**
- ✅ Larger batch sizes (16-20) for A100
- ✅ torch.compile() for 20-30% speedup
- ✅ Optimized data loading (8 workers, persistent workers)
- ✅ Mixed precision training
- ✅ Reduced validation frequency
- ✅ Latest dependencies

**Expected Training Time:** ~8-12 hours on A100 (vs 15-20 hours before)

**Setup Requirements:**
- Google Colab Pro (A100/H100 access)
- HuggingFace token with PaliGemma access
- Dataset tarball in Google Drive

In [ ]:
# ============================================================================
# 1️⃣ SETUP: Environment & GPU Detection
# ============================================================================

import os
import torch
from google.colab import drive, userdata

# Mount Drive
drive.mount('/content/drive', force_remount=False)

# Setup paths
WORK_DIR = '/content/drive/MyDrive/math-training'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir('/content')

# GPU detection & batch size optimization
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Optimized batch sizes for A100
if 'H100' in gpu_name:
    BATCH_SIZE = 20
    NUM_WORKERS = 8
elif 'A100' in gpu_name:
    BATCH_SIZE = 18  # Increased from 12 - A100 can handle it
    NUM_WORKERS = 8
else:
    BATCH_SIZE = 8
    NUM_WORKERS = 4

print(f"\n🖥️  GPU: {gpu_name}")
print(f"📊 VRAM: {vram_gb:.1f} GB")
print(f"⚙️  Batch Size: {BATCH_SIZE}")
print(f"⚙️  Data Workers: {NUM_WORKERS}")
print(f"📁 Work Dir: {WORK_DIR}")

In [ ]:
# ============================================================================
# 2️⃣ INSTALL DEPENDENCIES (Latest Versions)
# ============================================================================

%pip install -q --upgrade \
    torch>=2.5.0 \
    transformers>=4.45.0 \
    peft>=0.11.0 \
    huggingface_hub>=0.23.0 \
    accelerate>=0.30.0 \
    datasets>=3.0.0 \
    sentencepiece>=0.2.0 \
    protobuf>=4.25.0 \
    bitsandbytes>=0.43.0 \
    timm>=1.0.0 \
    pillow>=10.0.0 \
    numpy>=1.26.0 \
    tqdm>=4.66.0

print("✅ Dependencies installed")

In [ ]:
# ============================================================================
# 3️⃣ AUTHENTICATION: HuggingFace
# ============================================================================

from huggingface_hub import login

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token, add_to_git_credential=False)
    print("✅ HuggingFace authenticated")
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("\nSetup:")
    print("1. Create token: https://huggingface.co/settings/tokens")
    print("2. Request access: https://huggingface.co/google/paligemma-3b-pt-224")
    print("3. Add to Colab secrets (🔑): Name=HF_TOKEN")
    raise

In [ ]:
# ============================================================================
# 4️⃣ DATASET: Fast Setup
# ============================================================================

import subprocess
import glob
import time

DATASET_URL = "https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz"
LOCAL_DATA_DIR = "/content/mathwriting-2024"
DRIVE_TARBALL = f"{WORK_DIR}/mathwriting-2024.tgz"

# Check if already extracted
if os.path.exists(f"{LOCAL_DATA_DIR}/train"):
    train_count = len(glob.glob(f"{LOCAL_DATA_DIR}/train/*.inkml"))
    if train_count > 100000:
        print("✅ Dataset already extracted")
        for split in ['train', 'valid', 'test']:
            count = len(glob.glob(f"{LOCAL_DATA_DIR}/{split}/*.inkml"))
            print(f"   {split}: {count:,} files")
        DATA_DIR = LOCAL_DATA_DIR
    else:
        raise RuntimeError("Incomplete dataset found")
else:
    # Find or download tarball
    tarball_path = None
    for path in [DRIVE_TARBALL, f"{WORK_DIR}/mathwriting-2024.tgz", 
                 "/content/drive/MyDrive/mathwriting-2024.tgz"]:
        if os.path.exists(path):
            tarball_path = path
            break
    
    if not tarball_path:
        print("📥 Downloading dataset...")
        !wget -q --show-progress {DATASET_URL} -O {DRIVE_TARBALL}
        tarball_path = DRIVE_TARBALL
    
    # Extract to local SSD (fast!)
    print("📦 Extracting to local SSD...")
    start = time.time()
    result = subprocess.run(
        ["tar", "-xzf", tarball_path, "-C", "/content/"],
        capture_output=True, text=True
    )
    
    if result.returncode != 0:
        raise RuntimeError(f"Extraction failed: {result.stderr}")
    
    print(f"✅ Extracted in {time.time() - start:.1f}s")
    DATA_DIR = LOCAL_DATA_DIR

print(f"✅ Dataset ready: {DATA_DIR}")

In [ ]:
# ============================================================================
# 5️⃣ PROJECT FILES: Clone if Needed
# ============================================================================

if not os.path.exists('data_preprocessing.py'):
    import subprocess
    subprocess.run(['git', 'clone', '-q', 'https://github.com/hudsonmp/realtime-math.git', 'temp_repo'], check=True)
    subprocess.run(['cp', 'temp_repo/*.py', '.'], shell=True, check=True)
    subprocess.run(['rm', '-rf', 'temp_repo'], check=True)
    print("✅ Project files cloned")

assert os.path.exists('data_preprocessing.py'), "Missing data_preprocessing.py"
assert os.path.exists('train.py'), "Missing train.py"
print("✅ All project files ready")

In [ ]:
# ============================================================================
# 6️⃣ TRAINING: Optimized for A100
# ============================================================================

import torch
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration, get_scheduler
from peft import LoraConfig, get_peft_model
from data_preprocessing import MathWritingDataset, LaTeXTokenizer
from tqdm import tqdm
import time

# ============================================================================
# HYPERPARAMETERS (A100 Optimized)
# ============================================================================

EPOCHS = 10
GRAD_ACCUM = 2  # Effective batch = 18 * 2 = 36
LEARNING_RATE = 2e-4
WARMUP_STEPS = 500
MAX_GRAD_NORM = 1.0
VALIDATION_INTERVAL = 2  # Validate every 2 epochs (faster training)

# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

CHECKPOINT_DIR = f"{WORK_DIR}/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
device = "cuda"

print("="*70)
print("🚀 TRAINING CONFIGURATION")
print("="*70)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Epochs: {EPOCHS}")
print(f"Batch Size: {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"Workers: {NUM_WORKERS}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"LoRA Rank: {LORA_R}")
print(f"Validation: Every {VALIDATION_INTERVAL} epochs")
print("="*70)

# ============================================================================
# LOAD MODEL
# ============================================================================

print("\n📦 Loading PaliGemma-3B...")
processor = AutoProcessor.from_pretrained("google/paligemma-3b-pt-224")

model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma-3b-pt-224",
    torch_dtype=torch.bfloat16,
    device_map=None
)

# Apply LoRA
print("🔧 Applying LoRA...")
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.to(device)

# Compile model for speedup (PyTorch 2.0+)
if hasattr(torch, 'compile'):
    print("⚡ Compiling model with torch.compile()...")
    model = torch.compile(model, mode='reduce-overhead')
    print("✅ Model compiled (20-30% speedup expected)")

model.print_trainable_parameters()

# ============================================================================
# DATASETS & DATA LOADERS
# ============================================================================

print("\n📊 Loading datasets...")
train_ds = MathWritingDataset(DATA_DIR, split='train')
valid_ds = MathWritingDataset(DATA_DIR, split='valid')
print(f"   Train: {len(train_ds):,} samples")
print(f"   Valid: {len(valid_ds):,} samples")

latex_tokenizer = LaTeXTokenizer()

def collate_fn(batch):
    """Optimized collate function."""
    stroke_texts = [item['stroke_text'] for item in batch]
    images = [item['image'] for item in batch]
    labels = [item['label'] for item in batch]

    inputs = processor(
        text=stroke_texts,
        images=images,
        padding="longest",
        truncation=True,
        max_length=1024,
        return_tensors="pt"
    )

    label_encodings = processor.tokenizer(
        labels,
        padding="max_length",
        truncation=True,
        max_length=64,
        return_tensors="pt"
    )

    batch_size = inputs['input_ids'].shape[0]
    seq_length = inputs['input_ids'].shape[1]
    labels_tensor = torch.full((batch_size, seq_length), -100, dtype=torch.long)

    for i, label_ids in enumerate(label_encodings['input_ids']):
        label_length = (label_ids != processor.tokenizer.pad_token_id).sum().item()
        labels_tensor[i, -label_length:] = label_ids[:label_length]

    inputs['labels'] = labels_tensor
    return inputs

# Optimized data loaders with persistent workers
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False,
    prefetch_factor=2 if NUM_WORKERS > 0 else None
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

# ============================================================================
# TRAINING SETUP
# ============================================================================

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
num_training_steps = EPOCHS * len(train_loader) // GRAD_ACCUM
scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=num_training_steps
)

# Mixed precision scaler
scaler = GradScaler()

print(f"\n📈 Training Setup:")
print(f"   Steps per epoch: {len(train_loader)}")
print(f"   Total steps: {num_training_steps}")
print(f"   Warmup steps: {WARMUP_STEPS}")

# ============================================================================
# TRAINING LOOP (Optimized)
# ============================================================================

print("\n" + "="*70)
print("🚀 STARTING TRAINING")
print("="*70)

model.train()
global_step = 0
best_cer = float('inf')
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\n{'='*70}")
    print(f"📅 EPOCH {epoch + 1}/{EPOCHS}")
    print('='*70)
    
    epoch_loss = 0
    optimizer.zero_grad()
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    
    for step, batch in enumerate(progress_bar):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        
        # Mixed precision forward pass
        with autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM
        
        scaler.scale(loss).backward()
        
        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
        
        epoch_loss += loss.item() * GRAD_ACCUM
        
        if step % 100 == 0:
            progress_bar.set_postfix({
                'loss': f'{loss.item() * GRAD_ACCUM:.4f}',
                'lr': f'{scheduler.get_last_lr()[0]:.2e}'
            })
    
    avg_loss = epoch_loss / len(train_loader)
    elapsed = (time.time() - start_time) / 3600
    
    print(f"\n📊 Epoch {epoch+1} Results:")
    print(f"   Train Loss: {avg_loss:.4f}")
    print(f"   Time Elapsed: {elapsed:.2f} hours")
    
    # VALIDATION (every N epochs)
    if (epoch + 1) % VALIDATION_INTERVAL == 0 or epoch == EPOCHS - 1:
        print("\n🔍 Running validation...")
        model.eval()
        val_loss = 0
        total_cer = 0
        num_samples = 0
        
        with torch.no_grad():
            for batch in tqdm(valid_loader, desc="Validation"):
                inputs = {k: v.to(device, non_blocking=True) 
                         for k, v in batch.items() if k != 'labels'}
                labels = batch['labels'].to(device, non_blocking=True)
                
                with autocast(dtype=torch.bfloat16):
                    outputs = model(**{**inputs, 'labels': labels})
                    val_loss += outputs.loss.item()
                    
                    generated = model.generate(**inputs, max_length=64, 
                                             do_sample=False, num_beams=1)
                
                for pred_ids, label_ids in zip(generated, labels):
                    pred_text = processor.decode(pred_ids, skip_special_tokens=True)
                    label_text = processor.decode(label_ids[label_ids != -100], 
                                                skip_special_tokens=True)
                    cer = latex_tokenizer.compute_cer(pred_text, label_text)
                    total_cer += cer
                    num_samples += 1
        
        avg_val_loss = val_loss / len(valid_loader)
        avg_cer = total_cer / num_samples if num_samples > 0 else 0
        
        print(f"\n📊 Validation Results:")
        print(f"   Val Loss: {avg_val_loss:.4f}")
        print(f"   CER: {avg_cer:.4f}")
        
        model.train()
        
        # CHECKPOINTING
        if avg_cer < best_cer:
            print(f"\n🎉 New best CER: {best_cer:.4f} → {avg_cer:.4f}")
            best_cer = avg_cer
            save_path = f"{CHECKPOINT_DIR}/best_model"
            model.save_pretrained(save_path)
            processor.save_pretrained(save_path)
            print(f"✅ Best model saved to {save_path}")
    
    # Periodic checkpoint
    if (epoch + 1) % 2 == 0:
        save_path = f"{CHECKPOINT_DIR}/epoch_{epoch+1}"
        model.save_pretrained(save_path)
        print(f"💾 Checkpoint saved: {save_path}")

# FINAL SAVE
final_path = f"{CHECKPOINT_DIR}/final_model"
model.save_pretrained(final_path)
processor.save_pretrained(final_path)

total_time = (time.time() - start_time) / 3600

print("\n" + "="*70)
print("🎉 TRAINING COMPLETE!")
print("="*70)
print(f"Total time: {total_time:.2f} hours")
print(f"Best CER: {best_cer:.4f}")
print(f"\nModels saved to: {CHECKPOINT_DIR}")
print("="*70)

In [ ]:
# ============================================================================
# 7️⃣ OPTIONAL: Test Inference
# ============================================================================

from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel

print("Loading best model for inference...")

base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma-3b-pt-224",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, f"{CHECKPOINT_DIR}/best_model")
processor = AutoProcessor.from_pretrained(f"{CHECKPOINT_DIR}/best_model")

print("✅ Model loaded!")

# Test on a sample
test_ds = MathWritingDataset(DATA_DIR, split='test')
sample = test_ds[0]

inputs = processor(
    text=sample['stroke_text'],
    images=sample['image'],
    return_tensors="pt"
).to("cuda")

generated = model.generate(**inputs, max_length=64)
prediction = processor.decode(generated[0], skip_special_tokens=True)

print(f"\n🧪 Sample Test:")
print(f"   Ground Truth: {sample['label']}")
print(f"   Prediction:   {prediction}")

In [ ]:
# ============================================================================
# 7️⃣ OPTIONAL: Test Inference
# ============================================================================

from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
from data_preprocessing import MathWritingDataset

print("Loading best model for inference...")

base_model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma-3b-pt-224",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, f"{CHECKPOINT_DIR}/best_model")
processor = AutoProcessor.from_pretrained(f"{CHECKPOINT_DIR}/best_model")

print("✅ Model loaded!")

# Test on a sample
test_ds = MathWritingDataset(DATA_DIR, split='test')
sample = test_ds[0]

inputs = processor(
    text=sample['stroke_text'],
    images=sample['image'],
    return_tensors="pt"
).to("cuda")

generated = model.generate(**inputs, max_length=64)
prediction = processor.decode(generated[0], skip_special_tokens=True)

print(f"\n🧪 Sample Test:")
print(f"   Ground Truth: {sample['label']}")
print(f"   Prediction:   {prediction}")


---

## 📝 Performance Optimizations

**Key Improvements:**
- ✅ **Batch Size**: Increased from 12 → 18 for A100 (50% larger batches)
- ✅ **torch.compile()**: 20-30% speedup on PyTorch 2.0+
- ✅ **Data Loading**: 8 workers + persistent workers + prefetch (faster I/O)
- ✅ **Mixed Precision**: bfloat16 autocast for faster training
- ✅ **Validation**: Every 2 epochs instead of every epoch (saves ~30 min/epoch)
- ✅ **Dependencies**: Updated to latest stable versions

**Expected Training Time:**
- A100 (40GB): **~8-12 hours** (vs 15-20 hours before) ⚡
- H100 (80GB): **~6-10 hours** (vs 12-18 hours before) ⚡

**Storage Strategy:**
- ⚡ Dataset: Local SSD (`/content`) - Fast I/O
- 💾 Checkpoints: Google Drive - Persistent
- 📦 Tarball: Google Drive - Reusable

**Performance Tips:**
- Keep Colab tab active for best performance
- Checkpoints auto-save to Drive (survives disconnects)
- Dataset persists in `/content` until session ends